In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))
Path.cwd()

PosixPath('/home/avash/Documents/research/transformer-pt-analysis-main/src/Analysis')

# The Lazy Math

Everything in the universe wants to be in the lowest possible energy state.
A system is unstable if its energy is high, similar to Bohr's atomic model.

**Intuition:** Let's understand this in terms of a superconductor.


- **"Normal"** refers to the baseline energy of the system, as if the metal were not in a superconducting state.
- **Penalty 1:** The cost of creating a superconducting state.
- **Penalty 2:** If the superconducting state changes sharply from one spot to another, it costs energy.
  The superconductor prefers to remain smooth and uniform — this is the cost of changing from place to place.
- **Penalty 3:** The superconductor expels the magnetic field (Meissner effect).

**What about phase transition?**

In [13]:
import os
import torch
import analysis as a
from mintrans_clean import *

import importlib

importlib.reload(a)

# ---------------------------------------------------------------------------
# Empirical Structure Factor
# ---------------------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

files = os.listdir('./skeletons_var_lanbda')

path_wd = [{ "wd": decay.split('wd')[-1 :][0].split('_')[-1:][0].split('.pth')[0], "path": decay }for decay in files]

empirical_structure_factor = [] # Across 6 different samples

torch.set_printoptions(precision=4)

# ---------------------------------------------------------------------------
# Empirical Structure Factor across different weight decays
# ---------------------------------------------------------------------------


cfg = TrainConfig()

for obj in path_wd:
    path = os.path.join("skeletons_var_lanbda", obj['path'])
    weight_decay = obj['wd']

    model = MinimalTransformer(
        vocab_size=cfg.vocab_size, d_model=cfg.d_model, n_heads=cfg.n_heads,
        num_layers=cfg.num_layers, max_seq_len=cfg.seq_len,
        hidden_mlp=cfg.hidden_mlp,
    ).to(device)

    skeletons = torch.load(path, map_location=device)

    # Logits
    L = a.collect_state_logits(model, n=cfg.p, seq_len=cfg.seq_len,
        device=device, batch_size=cfg.batch_size,
        eq_token=cfg.p)

    empirical_structure_factor.append(a.oz_sf(L)[0])


kappa = 1.0
u = 1.0
r = 1.0
# empirical_structure_factor = 6 different samples
gl = [a.gl_energy(esf, r, kappa, u) for esf in empirical_structure_factor]
print(gl)


[tensor(3.8908e+21, device='cuda:0'), tensor(7.3033e+22, device='cuda:0'), tensor(5.3225e+23, device='cuda:0'), tensor(1.3011e+23, device='cuda:0'), tensor(8.1926e+22, device='cuda:0'), tensor(6.0454e+22, device='cuda:0')]
